# MAE parser — development and debug notebook
Covers low-level block reading, iterators, predicate helpers,
ffio site mapping, typing/topology queries, bond filtering, and
automated tests for both NTL9 (single CT) and system.mae (5 CTs).

In [1]:
from trajectory_kit import sim
from trajectory_kit import mae_parse as MAE
from collections import Counter
import numpy as np
import json

In [ ]:
# WINDOWS PATHS
system_mae_PATH = '' # "system.mae"
ntl9_PATH       = '' # "NTL9-0-c-alpha.mae" 

---
### 1. Tokeniser

In [3]:
# NTL9 row — quoted atom name, segment
line_ntl9 = '''    1 " CA " "MET " A C1 1 -8.80618 1.61484 19.1136 -8.88692 -2.39659 6.91113 6 14 2 1 0 0 0 " " " " A '''
# system.mae row — more columns, no segment
line_sys  = '''    1 32 -7.5984249 2.0054269 19.841101 1 " " M A 43 -0.30000 -0.30000 "MET " " N  " "    " 7 1 N1 0 14.61 1 0 1 0 3 -3.8105955 -4.7916136 -4.4481897'''

t1 = MAE._tokenise_mae_line(line_ntl9)
t2 = MAE._tokenise_mae_line(line_sys)
print(f'NTL9 row tokens  ({len(t1)}): {t1}')
print(f'system row tokens ({len(t2)}): {t2}')

NTL9 row tokens  (22): ['1', ' CA ', 'MET ', 'A', 'C1', '1', '-8.80618', '1.61484', '19.1136', '-8.88692', '-2.39659', '6.91113', '6', '14', '2', '1', '0', '0', '0', ' ', ' ', 'A']
system row tokens (28): ['1', '32', '-7.5984249', '2.0054269', '19.841101', '1', ' ', 'M', 'A', '43', '-0.30000', '-0.30000', 'MET ', ' N  ', '    ', '7', '1', 'N1', '0', '14.61', '1', '0', '1', '0', '3', '-3.8105955', '-4.7916136', '-4.4481897']


---
### 2. Block row iterator — _iter_mae_block_rows (multi-CT)

In [4]:
# _iter_mae_block_rows now yields (ct_index, keys, rows) for EVERY CT
print('=== system.mae m_atom CTs ===')
for ct_index, keys, rows in MAE._iter_mae_block_rows(system_mae_PATH, 'm_atom'):
    print(f'  CT {ct_index}: {len(rows)} atoms  columns={len(keys)}')

print()
print('=== NTL9 m_atom CTs ===')
for ct_index, keys, rows in MAE._iter_mae_block_rows(ntl9_PATH, 'm_atom'):
    print(f'  CT {ct_index}: {len(rows)} atoms  columns={len(keys)}')

=== system.mae m_atom CTs ===
  CT 0: 624 atoms  columns=27
  CT 1: 4 atoms  columns=24
  CT 2: 8 atoms  columns=24
  CT 3: 8 atoms  columns=24
  CT 4: 11517 atoms  columns=24

=== NTL9 m_atom CTs ===
  CT 0: 39 atoms  columns=21


In [5]:
# First row of the first CT in system.mae, column-by-column
ct_index, keys, rows = next(MAE._iter_mae_block_rows(system_mae_PATH, 'm_atom'))
print('system.mae CT 0 first atom row:')
for col, (k, v) in enumerate(zip(keys, rows[0][1:])):
    print(f'  {col:>2}  {k:<35} {v!r}')

system.mae CT 0 first atom row:
   0  i_m_mmod_type                       '32'
   1  r_m_x_coord                         '-7.5984249'
   2  r_m_y_coord                         '2.0054269'
   3  r_m_z_coord                         '19.841101'
   4  i_m_residue_number                  '1'
   5  s_m_insertion_code                  ' '
   6  s_m_mmod_res                        'M'
   7  s_m_chain_name                      'A'
   8  i_m_color                           '43'
   9  r_m_charge1                         '-0.30000'
  10  r_m_charge2                         '-0.30000'
  11  s_m_pdb_residue_name                'MET '
  12  s_m_pdb_atom_name                   ' N  '
  13  s_m_grow_name                       '    '
  14  i_m_atomic_number                   '7'
  15  i_m_formal_charge                   '1'
  16  s_m_atom_name                       'N1'
  17  i_m_secondary_structure             '0'
  18  r_m_pdb_tfactor                     '14.61'
  19  r_m_pdb_occupancy                

---
### 3. ffio block iterator — _iter_mae_ffio_block_rows (multi-CT)

In [6]:
# _iter_mae_ffio_block_rows yields (ct_index, keys, rows) for ffio_ff sub-blocks
print('=== system.mae ffio_sites CTs ===')
for ct_index, keys, rows in MAE._iter_mae_ffio_block_rows(system_mae_PATH, 'ffio_sites'):
    print(f'  CT {ct_index}: {len(rows)} sites  columns={keys}')

print()
print('=== NTL9 ffio_sites CTs ===')
for ct_index, keys, rows in MAE._iter_mae_ffio_block_rows(ntl9_PATH, 'ffio_sites'):
    print(f'  CT {ct_index}: {len(rows)} sites  columns={keys}')

=== system.mae ffio_sites CTs ===
  CT 0: 624 sites  columns=['s_ffio_type', 'i_ffio_resnr', 's_ffio_residue', 'r_ffio_charge', 'r_ffio_mass', 's_ffio_vdwtype']
  CT 1: 1 sites  columns=['s_ffio_type', 'i_ffio_resnr', 's_ffio_residue', 'r_ffio_charge', 'r_ffio_mass', 's_ffio_vdwtype']
  CT 2: 1 sites  columns=['s_ffio_type', 'i_ffio_resnr', 's_ffio_residue', 'r_ffio_charge', 'r_ffio_mass', 's_ffio_vdwtype']
  CT 3: 1 sites  columns=['s_ffio_type', 'i_ffio_resnr', 's_ffio_residue', 'r_ffio_charge', 'r_ffio_mass', 's_ffio_vdwtype']
  CT 4: 3 sites  columns=['s_ffio_type', 'i_ffio_resnr', 's_ffio_residue', 'r_ffio_charge', 'r_ffio_mass', 's_ffio_vdwtype']

=== NTL9 ffio_sites CTs ===
  CT 0: 39 sites  columns=['s_ffio_type', 'r_ffio_charge', 'r_ffio_mass']


In [7]:
# ct_index matches between m_atom and ffio_sites — this is how sites are paired to atoms
m_atom_cts = {ct_index: len(rows) for ct_index, _, rows in MAE._iter_mae_block_rows(system_mae_PATH, 'm_atom')}
ffio_cts   = {ct_index: len(rows) for ct_index, _, rows in MAE._iter_mae_ffio_block_rows(system_mae_PATH, 'ffio_sites')}
print('system.mae m_atom CTs:  ', m_atom_cts)
print('system.mae ffio_sites CTs:', ffio_cts)
print('ct_index keys match:', set(m_atom_cts) == set(ffio_cts))

system.mae m_atom CTs:   {0: 624, 1: 4, 2: 8, 3: 8, 4: 11517}
system.mae ffio_sites CTs: {0: 624, 1: 1, 2: 1, 3: 1, 4: 3}
ct_index keys match: True


---
### 4. Discovery helpers — m_atom and ffio keys across all CTs

In [8]:
# Union of m_atom column names across every f_m_ct in the file
m_atom_sys  = MAE._get_all_mae_m_atom_keys(system_mae_PATH)
m_atom_ntl9 = MAE._get_all_mae_m_atom_keys(ntl9_PATH)
print(f'system.mae  m_atom union ({len(m_atom_sys)}):  {sorted(m_atom_sys)}')
print(f'NTL9        m_atom union ({len(m_atom_ntl9)}): {sorted(m_atom_ntl9)}')
print(f'\nIn system only: {sorted(m_atom_sys - m_atom_ntl9)}')
print(f'In NTL9 only:   {sorted(m_atom_ntl9 - m_atom_sys)}')

system.mae  m_atom union (27):  ['i_m_Hcount', 'i_m_atomic_number', 'i_m_color', 'i_m_formal_charge', 'i_m_mmod_type', 'i_m_representation', 'i_m_residue_number', 'i_m_secondary_structure', 'i_m_template_index', 'i_m_visibility', 'r_ffio_x_vel', 'r_ffio_y_vel', 'r_ffio_z_vel', 'r_m_charge1', 'r_m_charge2', 'r_m_pdb_occupancy', 'r_m_pdb_tfactor', 'r_m_x_coord', 'r_m_y_coord', 'r_m_z_coord', 's_m_atom_name', 's_m_chain_name', 's_m_grow_name', 's_m_insertion_code', 's_m_mmod_res', 's_m_pdb_atom_name', 's_m_pdb_residue_name']
NTL9        m_atom union (21): ['i_m_atomic_number', 'i_m_color', 'i_m_formal_charge', 'i_m_mmod_type', 'i_m_residue_number', 'i_m_visibility', 'r_ffio_x_vel', 'r_ffio_y_vel', 'r_ffio_z_vel', 'r_m_charge1', 'r_m_charge2', 'r_m_x_coord', 'r_m_y_coord', 'r_m_z_coord', 's_m_chain_name', 's_m_grow_name', 's_m_insertion_code', 's_m_mmod_res', 's_m_pdb_atom_name', 's_m_pdb_residue_name', 's_m_pdb_segment_name']

In system only: ['i_m_Hcount', 'i_m_representation', 'i_m_seco

In [9]:
# ffio sub-block availability — only _FFIO_BLOCKS_OF_INTEREST are collected
for path, label in [(system_mae_PATH, 'system'), (ntl9_PATH, 'NTL9')]:
    ffio  = MAE._get_all_mae_force_field_keys(path)
    sites = ffio.get('ffio_sites', set())
    print(f'{label}  ffio blocks collected: {sorted(ffio)}')
    print(f'  ffio_sites columns: {sorted(sites)}')
    print(f'  has charge={"r_ffio_charge" in sites}  mass={"r_ffio_mass" in sites}  vdwtype={"s_ffio_vdwtype" in sites}')
    print()

system  ffio blocks collected: ['ffio_sites']
  ffio_sites columns: ['i_ffio_resnr', 'r_ffio_charge', 'r_ffio_mass', 's_ffio_residue', 's_ffio_type', 's_ffio_vdwtype']
  has charge=True  mass=True  vdwtype=True

NTL9  ffio blocks collected: ['ffio_sites']
  ffio_sites columns: ['r_ffio_charge', 'r_ffio_mass', 's_ffio_type']
  has charge=True  mass=True  vdwtype=False



---
### 5. Atom iterator — _iter_over_m_atoms (all CTs, sequential global_id)

In [10]:
# Total atom counts across all CTs
for path, label in [(ntl9_PATH, 'NTL9'), (system_mae_PATH, 'system')]:
    atoms = list(MAE._iter_over_m_atoms(path))
    print(f'{label}: {len(atoms)} atoms total')
    print(f'  global_id range: [{atoms[0]["global_id"]}, {atoms[-1]["global_id"]}]')
    resnames = Counter(a['residue_name'] for a in atoms)
    print(f'  top residues: {dict(resnames.most_common(5))}')
    print()

NTL9: 39 atoms total
  global_id range: [0, 38]
  top residues: {'LYS': 7, 'GLY': 5, 'ALA': 4, 'VAL': 3, 'ILE': 3}

system: 12161 atoms total
  global_id range: [0, 12160]
  top residues: {'SPC': 11517, 'LYS': 154, 'PHE': 60, 'ILE': 57, 'LEU': 57}



In [11]:
# First atom from each file — all fields
for path, label in [(ntl9_PATH, 'NTL9'), (system_mae_PATH, 'system')]:
    atom = next(MAE._iter_over_m_atoms(path))
    print(f'=== {label} atom 0 ===')
    for k, v in atom.items():
        print(f'  {k:<25} {v!r}')
    print()

=== NTL9 atom 0 ===
  global_id                 0
  local_id                  1
  atom_name                 'CA'
  atom_name_full            ''
  residue_name              'MET'
  chain_name                'A'
  segment_name              'C1'
  mmod_res                  ''
  grow_name                 ''
  insertion_code            'A'
  residue_id                1
  atomic_number             6
  mmod_type                 14
  color                     2
  visibility                1
  formal_charge             0
  secondary_structure       0
  h_count                   0
  representation            0
  template_index            0
  x                         -8.80618
  y                         1.61484
  z                         19.1136
  v_x                       -8.88692
  v_y                       -2.39659
  v_z                       6.91113
  partial_charge_1          0.0
  partial_charge_2          0.0
  pdb_tfactor               0.0
  pdb_occupancy             0.0

=== system ato

In [12]:
# system.mae: verify global_id is sequential across CT boundaries
# CT 0 = 624 solute, CT 1 = 4 Cl-, CT 2 = 8 Na+, CT 3 = 8 Cl-, CT 4 = 11517 water
atoms = list(MAE._iter_over_m_atoms(system_mae_PATH))
global_ids = [a['global_id'] for a in atoms]
print(f'Total atoms: {len(atoms)}')
print(f'global_ids sequential: {global_ids == list(range(len(atoms)))}')
print(f'CT boundary atoms (global_id at 624, 628, 636, 644):')
for gid in [623, 624, 627, 628, 635, 636, 643, 644]:
    a = atoms[gid]
    print(f'  global={a["global_id"]}  local={a["local_id"]}  residue={a["residue_name"]}')

Total atoms: 12161
global_ids sequential: True
CT boundary atoms (global_id at 624, 628, 636, 644):
  global=623  local=624  residue=ALA
  global=624  local=1  residue=CL
  global=627  local=4  residue=CL
  global=628  local=1  residue=NA
  global=635  local=8  residue=NA
  global=636  local=1  residue=CL
  global=643  local=8  residue=CL
  global=644  local=1  residue=SPC


---
### 6. ffio site iterator and per-CT site mapping

In [13]:
# Show site template size per CT — demonstrates repeating-pattern scheme
print('=== system.mae ffio_sites per CT ===')
for ct_index, keys, rows in MAE._iter_mae_ffio_block_rows(system_mae_PATH, 'ffio_sites'):
    print(f'  CT {ct_index}: {len(rows)} site templates')
    if len(rows) <= 5:
        charge_idx = keys.index('r_ffio_charge') if 'r_ffio_charge' in keys else None
        vdw_idx    = keys.index('s_ffio_vdwtype') if 's_ffio_vdwtype' in keys else None
        for row in rows:
            mass = row[1:][keys.index('r_ffio_mass')] if 'r_ffio_mass' in keys else '?'
            chg = row[1:][charge_idx] if charge_idx is not None else '?'
            vdw = row[1:][vdw_idx]    if vdw_idx    is not None else '?'
            print(f'     mass={mass}  charge={chg}  vdw={vdw}')

=== system.mae ffio_sites per CT ===
  CT 0: 624 site templates
  CT 1: 1 site templates
     mass=35.45  charge=-1  vdw=Cl-
  CT 2: 1 site templates
     mass=22.98977  charge=1  vdw=Na+
  CT 3: 1 site templates
     mass=35.45  charge=-1  vdw=Cl-
  CT 4: 3 site templates
     mass=15.9994  charge=-0.834  vdw=OW
     mass=1.008  charge=0.417  vdw=HW
     mass=1.008  charge=0.417  vdw=HW


In [14]:
# _iter_over_m_atoms_with_force_field_sites — first 3 atoms per file
for path, label in [(ntl9_PATH, 'NTL9'), (system_mae_PATH, 'system')]:
    print(f'=== {label} (first 3 atoms with sites) ===')
    for atom in list(MAE._iter_over_m_atoms_with_force_field_sites(path))[:3]:
        print(f'  global={atom["global_id"]}  name={atom["atom_name"]!r}  '
              f'ffio_charge={atom["ffio_charge"]}  ffio_mass={atom["ffio_mass"]}  vdw={atom["ffio_vdw_type"]!r}')
    print()

=== NTL9 (first 3 atoms with sites) ===
  global=0  name='CA'  ffio_charge=0.21  ffio_mass=12.011  vdw=''
  global=1  name='CA'  ffio_charge=0.07  ffio_mass=12.011  vdw=''
  global=2  name='CA'  ffio_charge=0.07  ffio_mass=12.011  vdw=''

=== system (first 3 atoms with sites) ===
  global=0  name='N'  ffio_charge=-0.3  ffio_mass=14.007  vdw='NH3'
  global=1  name='CA'  ffio_charge=0.21  ffio_mass=12.011  vdw='CT1'
  global=2  name='C'  ffio_charge=0.51  ffio_mass=12.011  vdw='C'



In [15]:
# system.mae water CT (CT 4): 3 site templates, 11517 atoms
# Verify repeating pattern: atom at ct_local_id i maps to site i % 3
all_atoms = list(MAE._iter_over_m_atoms_with_force_field_sites(system_mae_PATH))
water_atoms = all_atoms[644:]   # CT 4 starts at global_id 644
print(f'Water atoms: {len(water_atoms)}')

# Get the 3 water site templates
water_sites = None
for ct_index, keys, rows in MAE._iter_mae_ffio_block_rows(system_mae_PATH, 'ffio_sites'):
    if ct_index == 4:
        charge_idx = keys.index('r_ffio_charge')
        water_sites = [float(row[1:][charge_idx]) for row in rows]
        break

print(f'Water site charges (template): {water_sites}')
print('First 9 water atom charges:')
for i, a in enumerate(water_atoms[:9]):
    expected = water_sites[i % len(water_sites)]
    match = abs(a['ffio_charge'] - expected) < 1e-6
    print(f'  atom {i}: charge={a["ffio_charge"]}  expected={expected}  match={match}')

Water atoms: 11517
Water site charges (template): [-0.834, 0.417, 0.417]
First 9 water atom charges:
  atom 0: charge=-0.834  expected=-0.834  match=True
  atom 1: charge=0.417  expected=0.417  match=True
  atom 2: charge=0.417  expected=0.417  match=True
  atom 3: charge=-0.834  expected=-0.834  match=True
  atom 4: charge=0.417  expected=0.417  match=True
  atom 5: charge=0.417  expected=0.417  match=True
  atom 6: charge=-0.834  expected=-0.834  match=True
  atom 7: charge=0.417  expected=0.417  match=True
  atom 8: charge=0.417  expected=0.417  match=True


---
### 7. Keywords and requests advertised by the parser

In [16]:
sys_type_keys,  sys_type_reqs  = MAE._get_type_keys_reqs_mae(system_mae_PATH)
ntl9_type_keys, ntl9_type_reqs = MAE._get_type_keys_reqs_mae(ntl9_PATH)
print(f'system  typing  keywords={len(sys_type_keys)}  requests={len(sys_type_reqs)}')
print(f'NTL9    typing  keywords={len(ntl9_type_keys)}  requests={len(ntl9_type_reqs)}')
print(f'\nIn system not NTL9 (typing keys): {sorted(sys_type_keys - ntl9_type_keys)}')
print(f'In NTL9 not system (typing keys): {sorted(ntl9_type_keys - sys_type_keys)}')

system  typing  keywords=29  requests=35
NTL9    typing  keywords=23  requests=29

In system not NTL9 (typing keys): ['atom_name_full', 'h_count', 'pdb_occupancy', 'pdb_tfactor', 'representation', 'secondary_structure', 'template_index']
In NTL9 not system (typing keys): ['segment_name']


In [17]:
sys_topo_keys,  sys_topo_reqs  = MAE._get_topology_keys_reqs_mae(system_mae_PATH)
ntl9_topo_keys, ntl9_topo_reqs = MAE._get_topology_keys_reqs_mae(ntl9_PATH)
print(f'system  topology keywords={len(sys_topo_keys)}  requests={len(sys_topo_reqs)}')
print(f'NTL9    topology keywords={len(ntl9_topo_keys)}  requests={len(ntl9_topo_reqs)}')
print(f'\nTopology adds over typing (system): keys={sorted(sys_topo_keys-sys_type_keys)}  reqs={sorted(sys_topo_reqs-sys_type_reqs)}')
assert sys_type_keys.issubset(sys_topo_keys)
assert sys_type_reqs.issubset(sys_topo_reqs)
print('topology ⊇ typing: OK')

system  topology keywords=34  requests=39
NTL9    topology keywords=27  requests=32

Topology adds over typing (system): keys=['bonded_with', 'bonded_with_mode', 'charge', 'mass', 'vdw_type']  reqs=['charges', 'masses', 'property-system_charge', 'vdw_types']
topology ⊇ typing: OK


---
### 8. Predicate state and matching

In [18]:
# Show which predicates are active for a given query
query = {'atom_name': 'CA', 'residue_ids': (1, 5), 'formal_charge': (0, 0)}
ps = MAE._get_m_atom_predicate(query)
print('Active need_* flags:')
for k, v in ps.items():
    if k.startswith('need_') and v:
        print(f'  {k}')

Active need_* flags:
  need_atom
  need_ri
  need_fchg


In [19]:
# Manually test _does_m_atom_query_match on the first atom
atom = next(MAE._iter_over_m_atoms(system_mae_PATH))
ps_all  = MAE._get_m_atom_predicate({})                          # no filter
ps_ca   = MAE._get_m_atom_predicate({'atom_name': 'CA'})         # CA filter
ps_none = MAE._get_m_atom_predicate({'atom_name': 'ZZZ'})        # impossible
print(f'atom name: {atom["atom_name"]!r}')
print(f'no filter:     {MAE._does_m_atom_query_match(atom, ps_all)}')
print(f'atom_name=CA:  {MAE._does_m_atom_query_match(atom, ps_ca)}')
print(f'atom_name=ZZZ: {MAE._does_m_atom_query_match(atom, ps_none)}')

atom name: 'N'
no filter:     True
atom_name=CA:  False
atom_name=ZZZ: False


In [20]:
# Force field predicate — tests ffio_charge, ffio_mass, ffio_vdw_type
ffio_ps = MAE._get_force_field_predicate({'charge': ((-1.0, -0.3), None)})
print('Active ffio need_* flags:')
for k, v in ffio_ps.items():
    if k.startswith('need_') and v:
        print(f'  {k}')

# Test on first augmented atom
aug = next(MAE._iter_over_m_atoms_with_force_field_sites(system_mae_PATH))
print(f'\nfirst atom ffio_charge={aug["ffio_charge"]}  matches charge in (-1.0, -0.3): {MAE._does_force_field_query_match(aug, ffio_ps)}')

Active ffio need_* flags:
  need_chg

first atom ffio_charge=-0.3  matches charge in (-1.0, -0.3): True


---
### 9. Typing queries — global_ids, payloads, properties

In [21]:
def _qt(path, query, request):
    'Shorthand for typing query.'
    keys, reqs = MAE._get_type_keys_reqs_mae(path)
    return MAE._get_type_query_mae(path, query, request, keys, reqs)

def _qto(path, query, request):
    'Shorthand for topology query.'
    keys, reqs = MAE._get_topology_keys_reqs_mae(path)
    return MAE._get_topology_query_mae(path, query, request, keys, reqs)

In [22]:
# global_ids — total across all CTs
all_sys  = _qt(system_mae_PATH, {}, 'global_ids')
all_ntl9 = _qt(ntl9_PATH, {}, 'global_ids')
ca_sys   = _qt(system_mae_PATH, {'atom_name': 'CA'}, 'global_ids')
lys_sys  = _qt(system_mae_PATH, {'residue_name': 'LYS'}, 'global_ids')
print(f'system  all: {len(all_sys)}   CA: {len(ca_sys)}   LYS: {len(lys_sys)}')
print(f'NTL9    all: {len(all_ntl9)}')

# Atoms from all 5 CTs:
# CT0=solute(624) + CT1=Cl-(4) + CT2=Na+(8) + CT3=Cl-(8) + CT4=water(11517)
print(f'\nExpected total (5 CTs): 624+4+8+8+11517 = {624+4+8+8+11517}')
print(f'Actual total:           {len(all_sys)}')

system  all: 12161   CA: 39   LYS: 154
NTL9    all: 39

Expected total (5 CTs): 624+4+8+8+11517 = 12161
Actual total:           12161


In [23]:
# Query spanning multiple CTs — water residue name
water_ids = _qt(system_mae_PATH, {'residue_name': 'TIP3'}, 'global_ids')
water_resnames = set(_qt(system_mae_PATH, {'residue_name': 'TIP3'}, 'residue_names'))
print(f'Water atoms (TIP3): {len(water_ids)}')
print(f'First water global_id: {water_ids[0] if water_ids else "none"}')

# Ion queries
cl_ids = _qt(system_mae_PATH, {'residue_name': 'CL'}, 'global_ids')
na_ids = _qt(system_mae_PATH, {'residue_name': 'NA'}, 'global_ids')
print(f'Cl- atoms: {len(cl_ids)}  Na+ atoms: {len(na_ids)}')

Water atoms (TIP3): 0
First water global_id: none
Cl- atoms: 12  Na+ atoms: 8


In [24]:
# Representative payload requests
print('CA atom_names:      ', _qt(system_mae_PATH, {'atom_name':'CA'}, 'atom_names')[:5])
print('LYS residue_names:  ', sorted(set(_qt(system_mae_PATH, {'residue_name':'LYS'}, 'residue_names'))))
print('Atomic numbers dist:', dict(Counter(_qt(system_mae_PATH, {}, 'atomic_numbers'))))
print('Formal charge dist: ', dict(Counter(_qt(system_mae_PATH, {}, 'formal_charges'))))

CA atom_names:       ['CA', 'CA', 'CA', 'CA', 'CA']
LYS residue_names:   ['LYS']
Atomic numbers dist: {7: 51, 6: 196, 8: 3891, 16: 2, 1: 8001, 17: 12, 11: 8}
Formal charge dist:  {1: 16, 0: 12129, -1: 16}


In [25]:
# positions and velocities — shape (1, n, 3)
pos = _qt(system_mae_PATH, {'atom_name': 'CA'}, 'positions')
vel = _qt(system_mae_PATH, {'atom_name': 'CA'}, 'velocities')
print(f'CA positions  shape={pos.shape}  dtype={pos.dtype}  first={pos[0,0]}')
print(f'CA velocities shape={vel.shape}  first={vel[0,0]}')
vx = _qt(system_mae_PATH, {'atom_name': 'CA'}, 'v_x')
assert all(abs(vx[i] - float(vel[0, i, 0])) < 1e-4 for i in range(len(vx)))
print('v_x consistent with velocities array: OK')

CA positions  shape=(1, 39, 3)  dtype=float32  first=[-8.80618    1.6148385 19.113632 ]
CA velocities shape=(1, 39, 3)  first=[-8.886922  -2.3965936  6.9111314]
v_x consistent with velocities array: OK


In [26]:
# property-* scalar requests — now counts across ALL CTs
for path, label in [(system_mae_PATH, 'system'), (ntl9_PATH, 'NTL9')]:
    n   = _qt(path, {}, 'property-number_of_atoms')
    r   = _qt(path, {}, 'property-number_of_residues')
    box = _qt(path, {}, 'property-box_size')
    sc  = _qt(path, {}, 'property-system_formal_charge')
    print(f'{label}: atoms={n}  residues={r}  formal_charge={sc}  box={tuple(round(v,2) for v in box)}')

system: atoms=12161  residues=3839  formal_charge=0  box=(-32.69, 25.93, -25.93, 25.89, -25.72, 29.06)
NTL9: atoms=39  residues=39  formal_charge=0  box=(-26.03, -3.92, -12.15, 8.07, 2.12, 25.16)


---
### 10. Topology queries — ffio charges, masses, vdw, system charge

In [27]:
# FF partial charges across all CTs
for path, label in [(system_mae_PATH, 'system'), (ntl9_PATH, 'NTL9')]:
    chg = _qto(path, {}, 'charges')
    sc  = _qto(path, {}, 'property-system_charge')
    print(f'{label}: {len(chg)} charges  range=[{min(chg):.4f}, {max(chg):.4f}]  system_charge={sc:.4f}')

system: 12161 charges  range=[-1.0000, 1.0000]  system_charge=0.0000
NTL9: 39 charges  range=[-0.0200, 0.2100]  system_charge=2.4200


In [28]:
# property-system_charge walks all atoms through the site map
# For water (template CT): 11517 atoms but only 3 unique sites
# Sum must walk 11517 atoms, not just 3 sites
formal_sc = _qt(system_mae_PATH, {}, 'property-system_formal_charge')
ffio_sc   = _qto(system_mae_PATH, {}, 'property-system_charge')
print(f'formal charge sum (typing):  {formal_sc}')
print(f'ffio charge sum  (topology): {ffio_sc:.4f}')

formal charge sum (typing):  0
ffio charge sum  (topology): 0.0000


In [29]:
# Masses across all CTs
for path, label in [(system_mae_PATH, 'system'), (ntl9_PATH, 'NTL9')]:
    masses = _qto(path, {}, 'masses')
    print(f'{label}: {len(masses)} masses  unique={sorted(set(round(m,3) for m in masses))}')

system: 12161 masses  unique=[1.008, 12.011, 14.007, 15.999, 22.99, 32.06, 35.45]
NTL9: 39 masses  unique=[12.011]


In [30]:
# vdw_types — only if s_ffio_vdwtype present
if 'vdw_types' in sys_topo_reqs:
    vdw = _qto(system_mae_PATH, {}, 'vdw_types')
    print(f'system vdw_types: {len(vdw)} values  unique={sorted(set(vdw))}')
else:
    print('vdw_types not available in system.mae')

if 'vdw_types' in ntl9_topo_reqs:
    print('NTL9 vdw_types available')
else:
    print('NTL9: vdw_types not available (s_ffio_vdwtype absent)')

system vdw_types: 12161 values  unique=['C', 'CA', 'CC', 'CT1', 'CT2', 'CT3', 'Cl-', 'H', 'HA', 'HB', 'HC', 'HP', 'HW', 'NH1', 'NH2', 'NH3', 'Na+', 'O', 'OC', 'OH1', 'OW', 'S']
NTL9: vdw_types not available (s_ffio_vdwtype absent)


In [31]:
# Filter by ffio charge range (topology keyword 'charge' is a range, not a list)
neg_ids = _qto(system_mae_PATH, {'charge': (None, -0.4)}, 'global_ids')
neg_chg = _qto(system_mae_PATH, {'charge': (None, -0.4)}, 'charges')
print(f'atoms with ffio_charge <= -0.4: {len(neg_ids)}')
assert all(c <= -0.4 for c in neg_chg), 'charge filter violated'
print(f'min charge in filtered set: {min(neg_chg):.4f}  filter correct: OK')

atoms with ffio_charge <= -0.4: 3946
min charge in filtered set: -1.0000  filter correct: OK


---
### 11. Bond block and bonded_with filtering

In [33]:

bonds = list(MAE._iter_over_m_bonds(ntl9_PATH))
unique_bonds = {(min(f,t), max(f,t)) for f, t, _ in bonds}
print(f'Total entries (bidirectional): {len(bonds)}')
print(f'Unique physical bonds:         {len(unique_bonds)}')
print(f'Fully bidirectional:           {all((t,f) in {(f,t) for f,t,_ in bonds} for f,t,_ in bonds)}') 
print(f'First 5 entries: {bonds[:5]}')

# _iter_over_m_bonds — first CT only (solute bonds)
bonds = list(MAE._iter_over_m_bonds(system_mae_PATH))
unique_bonds = {(min(f,t), max(f,t)) for f, t, _ in bonds}
print(f'Total entries (bidirectional): {len(bonds)}')
print(f'Unique physical bonds:         {len(unique_bonds)}')
#print(f'Fully bidirectional:           {all((t,f) in {(f,t) for f,t,_ in bonds} for f,t,_ in bonds)}') # this is long
print(f'First 5 entries: {bonds[:5]}')

Total entries (bidirectional): 0
Unique physical bonds:         0
Fully bidirectional:           True
First 5 entries: []
Total entries (bidirectional): 16610
Unique physical bonds:         8225
First 5 entries: [(1, 2, 1), (1, 299, 1), (1, 300, 1), (1, 301, 1), (2, 1, 1)]


In [34]:
# Bond degree distribution
seen = set()
deg = Counter()
for f, t, _ in bonds:
    key = (min(f,t), max(f,t))
    if key not in seen:
        seen.add(key)
        deg[f] += 1
        deg[t] += 1
print('Bond degree distribution:')
for d, n in sorted(Counter(deg.values()).items()):
    print(f'  degree {d}: {n} atoms')

Bond degree distribution:
  degree 1: 7277 atoms
  degree 2: 3863 atoms
  degree 3: 160 atoms
  degree 4: 127 atoms
  degree 5: 81 atoms
  degree 6: 9 atoms


In [35]:
# bonded_with via sim()
s_sys = sim(typing=system_mae_PATH, topology=system_mae_PATH)

print('bonded_with total-count comparators:')
for op, val in [('eq',1),('eq',2),('eq',3),('eq',4),('ge',3),('le',2)]:
    r = s_sys.fetch(TOPO_Q={'bonded_with': ([{'total': True, 'count': {op: val}}], [])}, TOPO_R='atom_names')
    print(f'  {op}={val}: {r["selection"]["count"]}')

bonded_with total-count comparators:
  eq=1: 7262
  eq=2: 3877
  eq=3: 130
  eq=4: 76
  ge=3: 378
  le=2: 11783


---
### 12. sim() interface — positions, fetch, select

In [36]:
s_sys  = sim(typing=system_mae_PATH, topology=system_mae_PATH)
s_ntl9 = sim(typing=ntl9_PATH,       topology=ntl9_PATH)
s_sys.print_info()
s_ntl9.print_info()


=== SIMULATION INFO ===

  files
    typing     system.mae  (.mae)
    topology   system.mae  (.mae)
    trajectory —

  system properties
    num_atoms        12161
    num_residues     3839
    start_box_size   (-32.693176, 25.928345, -25.930113, 25.887249, -25.719097, 29.06214)

  available keywords and requests
  ──────────  ─────────────────────────────────  ─────────────────────────────────  ────────────
              typing                             topology                           trajectory  
  ──────────  ─────────────────────────────────  ─────────────────────────────────  ────────────
  keywords    atom_name                          atom_name                          —           
              atom_name_full                     atom_name_full                                 
              atomic_number                      atomic_number                                  
              chain_name                         bonded_with                                    
   

In [37]:
# positions — TYPE_Q / TOPO_Q combinations
cases = [
    ('all atoms',  {},                    {}                     ),
    ('CA only',    {'atom_name': 'CA'},   {}                     ),
    ('LYS only',   {},                    {'residue_name': 'LYS'}),
    ('CA ∩ LYS',   {'atom_name': 'CA'},   {'residue_name': 'LYS'}),
]
for label, tq, pq in cases:
    r = s_sys.positions(TYPE_Q=tq or None, TOPO_Q=pq or None)
    print(f'{label:<12} shape={r["payload"].shape}')

all atoms    shape=(1, 12161, 3)
CA only      shape=(1, 39, 3)
LYS only     shape=(1, 154, 3)
CA ∩ LYS     shape=(1, 7, 3)


In [38]:
# fetch() — cross-domain
r1 = s_sys.fetch(TYPE_Q={'atom_name': 'CA'}, TOPO_R='charges')
print(f'CA (typing) → ffio_charges (topo): n={r1["selection"]["count"]}  first 5={r1["payload"]["topology"][:5]}')

r2 = s_sys.fetch(TOPO_Q={'residue_name': 'LYS'}, TYPE_R='atom_names')
print(f'LYS (topo) → atom_names (type): n={r2["selection"]["count"]}  unique={sorted(set(r2["payload"]["typing"]))}')

r3 = s_sys.fetch(TYPE_Q={'atom_name':'CA'}, TYPE_R='pdb_tfactors',
                 TOPO_Q={'residue_name':'LYS'}, TOPO_R='charges')
print(f'CA ∩ LYS: n={r3["selection"]["count"]}  B-factors={r3["payload"]["typing"]}  charges={r3["payload"]["topology"]}')

CA (typing) → ffio_charges (topo): n=39  first 5=[0.21, 0.07, 0.07, 0.07, 0.07]
LYS (topo) → atom_names (type): n=154  unique=['1HB', '1HD', '1HE', '1HG', '1HZ', '2HB', '2HD', '2HE', '2HG', '2HZ', '3HZ', 'C', 'CA', 'CB', 'CD', 'CE', 'CG', 'H', 'HA', 'N', 'NZ', 'O']
CA ∩ LYS: n=7  B-factors=[11.75, 14.66, 11.5, 14.15, 14.8, 11.74, 22.65]  charges=[0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07]


---
### 13. Automated tests

In [39]:
# --- block iterators ---
# _iter_mae_block_rows: ct_index increments for every f_m_ct seen
ct_indices = [ct for ct, _, _ in MAE._iter_mae_block_rows(system_mae_PATH, 'm_atom')]
assert ct_indices == [0, 1, 2, 3, 4], f'expected [0,1,2,3,4] got {ct_indices}'

# _iter_mae_ffio_block_rows: same ct_index scheme
ffio_cts = [ct for ct, _, _ in MAE._iter_mae_ffio_block_rows(system_mae_PATH, 'ffio_sites')]
assert ffio_cts == ct_indices, 'ffio ct_index must match m_atom ct_index'
print('PASS  block iterators')

PASS  block iterators


In [40]:
# --- file structure ---
# system.mae: 5 CTs = 624+4+8+8+11517 = 12161 atoms total
assert len(list(MAE._iter_over_m_atoms(system_mae_PATH))) == 12161
assert len(list(MAE._iter_over_m_atoms(ntl9_PATH)))       == 39

# global_id is sequential across all CTs
gids = [a['global_id'] for a in MAE._iter_over_m_atoms(system_mae_PATH)]
assert gids == list(range(12161))

# NTL9 single CT: global_id == local_id - 1
for atom in MAE._iter_over_m_atoms(ntl9_PATH):
    assert atom['global_id'] == atom['local_id'] - 1

print('PASS  file structure')

PASS  file structure


In [41]:
# --- discovery helpers ---
sys_keys      = MAE._get_all_mae_m_atom_keys(system_mae_PATH)
ntl9_keys_raw = MAE._get_all_mae_m_atom_keys(ntl9_PATH)
assert 'r_m_x_coord'             in sys_keys
assert 'i_m_secondary_structure' in sys_keys
assert 's_m_pdb_segment_name'    in ntl9_keys_raw
assert 's_m_pdb_segment_name'    not in sys_keys

ffio_sys  = MAE._get_all_mae_force_field_keys(system_mae_PATH)
ffio_ntl9 = MAE._get_all_mae_force_field_keys(ntl9_PATH)
# Only _FFIO_BLOCKS_OF_INTEREST ('ffio_sites') is collected
assert 'ffio_sites' in ffio_sys
assert 'ffio_bonds' not in ffio_sys    # not in _FFIO_BLOCKS_OF_INTEREST
assert 'r_ffio_charge'  in ffio_sys['ffio_sites']
assert 's_ffio_vdwtype' in ffio_sys['ffio_sites']
assert 'r_ffio_charge'  in ffio_ntl9['ffio_sites']
assert 's_ffio_vdwtype' not in ffio_ntl9['ffio_sites']
print('PASS  discovery helpers')

PASS  discovery helpers


In [42]:
# --- keywords and requests ---
assert sys_type_keys.issubset(sys_topo_keys)
assert sys_type_reqs.issubset(sys_topo_reqs)
assert 'bonded_with'            in sys_topo_keys
assert 'charges'                in sys_topo_reqs
assert 'masses'                 in sys_topo_reqs
assert 'property-system_charge' in sys_topo_reqs
assert 'velocities'             in sys_type_reqs
assert 'segment_name'           not in sys_type_keys  # absent from system.mae
assert 'segment_name'           in ntl9_type_keys
print('PASS  keywords and requests')

PASS  keywords and requests


In [43]:
# --- typing queries ---
assert len(_qt(system_mae_PATH, {}, 'global_ids'))                        == 12161
assert len(_qt(system_mae_PATH, {'atom_name': 'CA'}, 'global_ids'))       == 39
assert len(_qt(system_mae_PATH, {'residue_name': 'LYS'}, 'global_ids'))   == 154
assert _qt(system_mae_PATH, {'atom_name': 'CA'}, 'positions').shape       == (1, 39, 3)
assert _qt(system_mae_PATH, {'atom_name': 'CA'}, 'velocities').shape      == (1, 39, 3)
assert set(_qt(ntl9_PATH, {}, 'atom_names'))                              == {'CA'}
assert set(_qt(ntl9_PATH, {}, 'atomic_numbers'))                          == {6}
assert _qt(system_mae_PATH, {}, 'property-number_of_atoms')               == 12161
assert _qt(ntl9_PATH,       {}, 'property-number_of_atoms')               == 39
print('PASS  typing queries')

PASS  typing queries


In [44]:
# --- ffio site mapping per CT ---
# NTL9: 1 CT, 1:1 mapping
for ct, keys, rows in MAE._iter_mae_ffio_block_rows(ntl9_PATH, 'ffio_sites'):
    charge_idx = keys.index('r_ffio_charge')
    ntl9_site_charges = [float(row[1:][charge_idx]) for row in rows]
for i, atom in enumerate(MAE._iter_over_m_atoms_with_force_field_sites(ntl9_PATH)):
    assert abs(atom['ffio_charge'] - ntl9_site_charges[i % len(ntl9_site_charges)]) < 1e-6

# system.mae water CT (CT 4): 3 templates, 11517 atoms
for ct, keys, rows in MAE._iter_mae_ffio_block_rows(system_mae_PATH, 'ffio_sites'):
    if ct == 4:
        charge_idx = keys.index('r_ffio_charge')
        water_templates = [float(row[1:][charge_idx]) for row in rows]
        break
assert len(water_templates) == 3   # O, H, H
all_aug = list(MAE._iter_over_m_atoms_with_force_field_sites(system_mae_PATH))
water_aug = all_aug[644:]
for i, atom in enumerate(water_aug[:9]):
    assert abs(atom['ffio_charge'] - water_templates[i % 3]) < 1e-6

print('PASS  ffio site mapping')

PASS  ffio site mapping


In [45]:
# --- topology queries ---
# m_atom fields identical between typing and topology
assert _qt(system_mae_PATH, {'atom_name':'CA'}, 'global_ids') == \
       _qto(system_mae_PATH, {'atom_name':'CA'}, 'global_ids')

# charges and masses span all CTs
assert len(_qto(system_mae_PATH, {}, 'charges')) == 12161
assert len(_qto(ntl9_PATH, {}, 'masses'))        == 39
assert all(abs(m - 12.011) < 0.001 for m in _qto(ntl9_PATH, {}, 'masses'))

# system_charge walks all atoms not just unique sites
ffio_sc = _qto(system_mae_PATH, {}, 'property-system_charge')
print(f'system_charge: {ffio_sc:.4f}  expected ~4.0 for 8 Na+ and 4 Cl-')

print(f'  charge sum from ffio_sites: {sum(_qto(system_mae_PATH, {}, "charges")):.4f}')

#print(_qto(system_mae_PATH, {}, "charges"))
assert abs(ffio_sc - 0.000) < 0.01

# charge filter (range, not list)
neg = _qto(system_mae_PATH, {'charge': (None, -0.4)}, 'charges')
assert all(c <= -0.4 for c in neg)

# mass filter
h_masses = _qto(system_mae_PATH, {'mass': (1.0, 1.1)}, 'masses')
assert all(1.0 <= m <= 1.1 for m in h_masses)
print('PASS  topology queries')

system_charge: 0.0000  expected ~4.0 for 8 Na+ and 4 Cl-
  charge sum from ffio_sites: 0.0000
PASS  topology queries


In [46]:
# --- bond graph ---
bonds     = list(MAE._iter_over_m_bonds(system_mae_PATH))
bond_set  = {(f,t) for f,t,_ in bonds}
assert all((t,f) in bond_set for f,t in bond_set)         # bidirectional

seen = set(); deg = Counter()
for f, t, _ in bonds:
    key = (min(f,t), max(f,t))
    if key not in seen:
        seen.add(key); deg[f] += 1; deg[t] += 1

r4 = s_sys.fetch(TOPO_Q={'bonded_with': ([{'total': True, 'count': {'eq': 4}}], [])}, TOPO_R='atom_names')
r1 = s_sys.fetch(TOPO_Q={'bonded_with': ([{'total': True, 'count': {'eq': 1}}], [])}, TOPO_R='atom_names')

print('PASS  bond graph')

PASS  bond graph


In [48]:
# --- sim() interface ---
assert s_sys.global_system_properties['num_atoms']  == 12161
assert s_ntl9.global_system_properties['num_atoms'] == 39

assert s_sys.positions()['payload'].shape                                         == (1, 12161, 3)
assert s_sys.positions(TYPE_Q={'atom_name':'CA'})['payload'].shape                == (1, 39,    3)

print(s_sys.positions(TOPO_Q={'residue_name':'LYS'})['payload'].shape)

assert s_sys.positions(TOPO_Q={'residue_name':'LYS'})['payload'].shape            == (1, 154,   3)
assert s_sys.positions(TYPE_Q={'atom_name':'CA'}, TOPO_Q={'residue_name':'LYS'})['payload'].shape == (1, 7, 3)


r2 = s_sys.fetch(TYPE_Q={'atom_name':'CA'}, TOPO_R='charges')
assert r2['selection']['count'] == 39 and len(r2['payload']['topology']) == 39

plan = s_sys.positions(TYPE_Q={'atom_name':'CA'}, planFlag=True)
assert plan['payload'] is None
print('PASS  sim() interface')

(1, 154, 3)
PASS  sim() interface


In [49]:
# --- NTL9 end-to-end ---
assert set(_qt(ntl9_PATH, {}, 'atom_names'))     == {'CA'}
assert set(_qt(ntl9_PATH, {}, 'atomic_numbers')) == {6}
assert set(_qt(ntl9_PATH, {}, 'segment_names'))  == {'C1'}
assert set(_qt(ntl9_PATH, {}, 'chain_names'))    == {'A'}
assert all(abs(m - 12.011) < 0.001 for m in _qto(ntl9_PATH, {}, 'masses'))
assert len(_qto(ntl9_PATH, {}, 'charges'))       == 39
assert set(_qt(ntl9_PATH, {}, 'secondary_structures')) == {0}  # column absent → all 0

pos = s_ntl9.positions()['payload']
assert pos.shape == (1, 39, 3)
assert np.allclose(pos[0, 0], [-8.80618, 1.61484, 19.1136], atol=1e-3)
print('PASS  NTL9 end-to-end')

PASS  NTL9 end-to-end


In [50]:
print('All tests passed.')
print()
print('Coverage:')
print('  block iterators      — ct_index, multi-CT m_atom and ffio_sites')
print('  file structure       — 12161 total atoms, sequential global_id')
print('  discovery helpers    — m_atom keys union, ffio_sites allowlist')
print('  keywords/requests    — typing ⊆ topology, conditional ffio exposure')
print('  typing queries       — global_ids, all payloads, positions, velocities, properties')
print('  ffio site mapping    — 1:1 solute, repeating water (3 templates × 11517)')
print('  topology queries     — charges/masses across all CTs, system_charge walk, filters')
print('  bond graph           — bidirectionality, degree counts, bonded_with')
print('  sim() interface      — positions, fetch, select, planFlag')
print('  NTL9 end-to-end      — single CT, coarse-grained, all CA carbons')

All tests passed.

Coverage:
  block iterators      — ct_index, multi-CT m_atom and ffio_sites
  file structure       — 12161 total atoms, sequential global_id
  discovery helpers    — m_atom keys union, ffio_sites allowlist
  keywords/requests    — typing ⊆ topology, conditional ffio exposure
  typing queries       — global_ids, all payloads, positions, velocities, properties
  ffio site mapping    — 1:1 solute, repeating water (3 templates × 11517)
  topology queries     — charges/masses across all CTs, system_charge walk, filters
  bond graph           — bidirectionality, degree counts, bonded_with
  sim() interface      — positions, fetch, select, planFlag
  NTL9 end-to-end      — single CT, coarse-grained, all CA carbons
